# 1D Poisson Equation with a PINN

Solve the boundary value problem

$$-u''(x) = \pi^2 \sin(\pi x), \quad x \in [0, 1], \qquad u(0) = u(1) = 0$$

with a Physics-Informed Neural Network ([Raissi et al., 2019](https://arxiv.org/abs/1711.10561)).
The network is trained on the PDE residual and the boundary conditions as well as on data. The exact
solution $u^*(x) = \sin(\pi x)$ is used to measure the error.

In [ ]:
# Install keras-sciml, then choose a backend: "tensorflow", "torch" or "jax"
!pip install keras-sciml
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Hyperparameters

`N_COLLOCATION` is the number of grid points the residual is evaluated at.

In [ ]:
N_COLLOCATION = 128
HIDDEN_LAYERS = 3
HIDDEN_UNITS = 64
EPOCHS = 60
LEARNING_RATE = 0.01

## Collocation points

In [ ]:
import keras
import numpy as np
from keras import ops

from keras_sciml.models import PINN

keras.utils.set_random_seed(42)

x_colloc = np.linspace(0.0, 1.0, N_COLLOCATION, dtype=np.float32)[:, None]
u_exact = np.sin(np.pi * x_colloc)
dx = 1.0 / (N_COLLOCATION - 1)

## Physics residual

The residual $R(u, x) = -u'' - f$ uses a 3-point central difference for $u''$ at the interior points.
The boundary values $u(0)$ and $u(1)$ are added as extra residual terms with a weight of 10.

In [ ]:
def poisson_residual(u, x):
    d2u = (u[2:] - 2.0 * u[1:-1] + u[:-2]) / (dx ** 2)
    f = (np.pi ** 2) * ops.sin(np.pi * x[1:-1])
    interior = -d2u - f
    boundary = ops.stack([u[0], u[-1]], axis=0)
    return ops.concatenate([interior, 10.0 * boundary], axis=0)

## Define the model

Fourier features of the input coordinate help the MLP fit oscillating solutions. `pde_weight` scales
the residual term relative to the data loss.

In [ ]:
model = PINN(
    output_dim=1,
    hidden_layers=HIDDEN_LAYERS,
    hidden_units=HIDDEN_UNITS,
    activation="tanh",
    use_fourier_features=True,
    fourier_dim=16,
    residual_fn=poisson_residual,
    pde_weight=0.1,
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse",
    metrics=["mae"],
)

## Train

The whole grid is one batch, so the finite-difference stencil always sees neighbouring points.

In [ ]:
history = model.fit(
    x_colloc,
    u_exact,
    epochs=EPOCHS,
    batch_size=N_COLLOCATION,
    verbose=2,
)

## Evaluate

In [ ]:
preds = model.predict(x_colloc, verbose=0)
l2_error = np.linalg.norm(preds - u_exact) / np.linalg.norm(u_exact)
print(f"Relative L2 error to the exact solution: {l2_error:.4e}")